In [ ]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.windows import from_bounds
from pathlib import Path


def clip_country_raster(
    raster_path: Path,
    country_gdf: gpd.GeoDataFrame,
    output_dir: Path,
    max_pixels: int = 30_000_000,
):
    """Clip one raster to the country polygon while guarding against huge in-memory allocations."""
    filename_parts = raster_path.stem.split("_")
    if len(filename_parts) < 3:
        print(f"Skipping {raster_path.name}: filename format unrecognized.")
        return None

    country_code = filename_parts[2]
    country_slice = country_gdf[country_gdf["ISO_A3"] == country_code].copy()

    if country_slice.empty:
        print(f"  Warning: No features found for {country_code}. Skipping.")
        return None

    with rasterio.open(raster_path) as src:
        if country_slice.crs != src.crs:
            print(f"  Reprojecting {country_code} boundary to match raster CRS...")
            country_slice = country_slice.to_crs(src.crs)

        bounds = country_slice.total_bounds
        window = from_bounds(*bounds, transform=src.transform)
        width = int(window.width)
        height = int(window.height)
        pixel_count = width * height

        print(f"Processing {country_code}: approx. {pixel_count:,} pixels")
        if pixel_count > max_pixels:
            print(
                f"  Skipping {raster_path.name}: clipped window is too large "
                f"({pixel_count:,} > {max_pixels:,} pixels). "
                "Use smaller AOIs or tile the raster before masking."
            )
            return None

        geometries = country_slice.geometry.tolist()

        try:
            out_image, out_transform = mask(src, geometries, crop=True, all_touched=False)
        except ValueError as exc:
            print(f"  Error masking {country_code} ({raster_path.name}): {exc}")
            return None

        out_meta = src.meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
            "nodata": src.nodata if src.nodata is not None else 0,
        })

        out_file = output_dir / f"{raster_path.stem}_clipped.tif"
        with rasterio.open(out_file, "w", **out_meta) as dest:
            dest.write(out_image)

        print(f"  Successfully saved: {out_file.name}")
        return out_file


# 1. Define your paths
raster_dir = Path(r"C:\Users\AFahrezi\Documents\GitHub\generic_rs_code\raster_mosaicking\rasters")
master_shapefile_path = Path(r"C:\Users\AFahrezi\Documents\GitHub\generic_rs_code\raster_mosaicking\AOI\FAO_RCP51_Country.shp")
output_dir = Path(r"C:\Users\AFahrezi\Documents\GitHub\generic_rs_code\raster_mosaicking\clipped_output")
output_dir.mkdir(parents=True, exist_ok=True)

# 2. Load the master shapefile ONCE
print("Loading master shapefile...")
sea_gdf = gpd.read_file(master_shapefile_path)

# 3. Iterate through all raster files in the directory
for raster_path in sorted(raster_dir.glob("*.tif")):
    clip_country_raster(raster_path, sea_gdf, output_dir)

print("Batch clipping complete.")


Loading master shapefile...
Processing BTN...
  Reprojecting BTN boundary to match raster CRS...
  Successfully saved: cocoa_binary_BTN_2024_clipped.tif
Processing KHM...
  Reprojecting KHM boundary to match raster CRS...
  Successfully saved: cocoa_binary_KHM_2024_clipped.tif
Processing LAO...
  Reprojecting LAO boundary to match raster CRS...
  Successfully saved: cocoa_binary_LAO_2024_clipped.tif
Processing MMR...
  Reprojecting MMR boundary to match raster CRS...
  Successfully saved: cocoa_binary_MMR_2024_clipped.tif
Processing MYS...
  Reprojecting MYS boundary to match raster CRS...
  Successfully saved: cocoa_binary_MYS_2024_clipped.tif
Processing PHL...
  Reprojecting PHL boundary to match raster CRS...
  Successfully saved: cocoa_binary_PHL_2024_clipped.tif
Processing PNG...
  Reprojecting PNG boundary to match raster CRS...
  Successfully saved: cocoa_binary_PNG_2024_clipped.tif
Processing THA...
  Reprojecting THA boundary to match raster CRS...
  Successfully saved: cocoa_

In [ ]:
from pathlib import Path
import rasterio
from rasterio.merge import merge


def merge_rasters(folder: Path, output_path: Path, method: str = "max", nodata: int = 0):
    """Merge all TIFFs in a folder into one output raster and close files cleanly."""
    file_list = sorted(str(p) for p in folder.glob("*.tif"))
    print("Found files:", file_list)

    if not file_list:
        raise FileNotFoundError(f"No TIFF files found in {folder}")

    src_files_to_mosaic = [rasterio.open(fp) for fp in file_list]

    try:
        mosaic, out_transform = merge(src_files_to_mosaic, method=method, nodata=nodata)

        out_meta = src_files_to_mosaic[0].meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": mosaic.shape[1],
            "width": mosaic.shape[2],
            "transform": out_transform,
            "nodata": nodata,
            "compress": "lzw",
        })

        with rasterio.open(output_path, "w", **out_meta) as dest:
            dest.write(mosaic)

        print(f"Saved merged raster: {output_path}")
    finally:
        for src in src_files_to_mosaic:
            src.close()


# Example usage
folder = Path(r"C:\Users\AFahrezi\Documents\GitHub\generic_rs_code\raster_mosaicking\commodities_country\coffee_clip")
out_path = Path(r"C:\Users\AFahrezi\Documents\GitHub\generic_rs_code\raster_mosaicking\commodities_country\coffee_binary_combine_2024.tif")
merge_rasters(folder, out_path)


IndexError: list index out of range

In [ ]:
from pathlib import Path
import rasterio
from rasterio.merge import merge


def merge_rasters(folder: Path, output_path: Path, method: str = "max", nodata: int = 0):
    """Merge all TIFFs in a folder into one output raster and close files cleanly."""
    file_list = sorted(str(p) for p in folder.glob("*.tif"))
    print("Found files:", file_list)

    if not file_list:
        raise FileNotFoundError(f"No TIFF files found in {folder}")

    src_files_to_mosaic = [rasterio.open(fp) for fp in file_list]

    try:
        mosaic, out_transform = merge(src_files_to_mosaic, method=method, nodata=nodata)

        out_meta = src_files_to_mosaic[0].meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": mosaic.shape[1],
            "width": mosaic.shape[2],
            "transform": out_transform,
            "nodata": nodata,
            "compress": "lzw",
        })

        with rasterio.open(output_path, "w", **out_meta) as dest:
            dest.write(mosaic)

        print(f"Merged raster saved to: {output_path}")
    finally:
        for src in src_files_to_mosaic:
            src.close()


folder = Path(r"C:\Users\AFahrezi\Documents\GitHub\generic_rs_code\raster_mosaicking\commodities_country\coffee_clip")
out_path = Path(r"C:\Users\AFahrezi\Documents\GitHub\generic_rs_code\raster_mosaicking\commodities_country\coffee_binary_combine_2024.tif")
merge_rasters(folder, out_path)


Found files: ['C:\\Users\\AFahrezi\\Documents\\GitHub\\generic_rs_code\\raster_mosaicking\\commodities_country\\coffee_clip\\coffee_binary_BTN_2024_clipped.tif', 'C:\\Users\\AFahrezi\\Documents\\GitHub\\generic_rs_code\\raster_mosaicking\\commodities_country\\coffee_clip\\coffee_binary_IDN_2024_clipped.tif', 'C:\\Users\\AFahrezi\\Documents\\GitHub\\generic_rs_code\\raster_mosaicking\\commodities_country\\coffee_clip\\coffee_binary_IDN_KAL_2024_clipped.tif', 'C:\\Users\\AFahrezi\\Documents\\GitHub\\generic_rs_code\\raster_mosaicking\\commodities_country\\coffee_clip\\coffee_binary_KHM_2024_clipped.tif', 'C:\\Users\\AFahrezi\\Documents\\GitHub\\generic_rs_code\\raster_mosaicking\\commodities_country\\coffee_clip\\coffee_binary_LAO_2024_clipped.tif', 'C:\\Users\\AFahrezi\\Documents\\GitHub\\generic_rs_code\\raster_mosaicking\\commodities_country\\coffee_clip\\coffee_binary_MMR_2024_clipped.tif', 'C:\\Users\\AFahrezi\\Documents\\GitHub\\generic_rs_code\\raster_mosaicking\\commodities_count

In [ ]:
# This cell is intentionally left as a final write step when you want to save a custom output.
# The merge function already writes the file in the previous cell, so this cell is optional.
print("Mosaic export complete.")
